# Reading PAW Data from HDF5

This notebook shows how to load and explore pre-computed PAW integrals using `PawReader`. We use the test data file `data/lcbo_2x2x2.h5`, which contains results from a 2x2x2 k-mesh LCBO calculation for metallic hydrogen.

**No GPAW installation is needed** --- the reader depends only on `numpy` and `h5py`.

> **Prerequisite:** The data file `data/lcbo_2x2x2.h5` must exist. It is generated by running notebook **02_gpaw_extraction** (or provided by a collaborator).

In [1]:
import numpy as np
from bloch_paw import PawReader

DATA_FILE = "../data/lcbo_2x2x2.h5"

## Loading the Data

`PawReader.load()` reads the full HDF5 file and returns the core arrays. The return values are:
- **k_mesh**: Cartesian k-point coordinates (Nk, 3) in Å⁻¹
- **L_size**: Supercell scaling factors (Lx, Ly, Lz)
- **a_vectors**: Primitive lattice vectors (3, 3) in Å
- **N_pw**: Number of plane waves (FFT grid points with G ≠ 0)
- **rho**: Smooth pseudo pair-density $\tilde{\rho}$ on the real-space FFT grid
- **Ca_dict**: PAW on-site Coulomb correction tensors $C^a$, one per atom
- **Da_dict**: Projector density matrices $D^a$, one per atom
- **meta**: Metadata dictionary (extraction parameters, timestamps, etc.)

In [2]:
reader = PawReader(DATA_FILE)
k_mesh, L_size, a_vectors, N_pw, rho, Ca_dict, Da_dict, meta = reader.load()

print(f"Number of k-points (Nk):  {k_mesh.shape[0]}")
print(f"Number of bands (Nb):     {rho.shape[1]}")
print(f"Supercell scaling:        {L_size}")
print(f"Number of plane waves:    {N_pw}")
print(f"Number of atoms:          {len(Ca_dict)}")

Number of k-points (Nk):  8
Number of bands (Nb):     5
Supercell scaling:        [1 1 1]
Number of plane waves:    511
Number of atoms:          1


## K-Point Mesh

The k-mesh is a Monkhorst-Pack grid in the full Brillouin zone. For a 2×2×2 supercell with Γ-centred sampling, we get 8 k-points.

In [3]:
print("K-point mesh (Cartesian, Å⁻¹):")
print(f"  Shape: {k_mesh.shape}")
for i, k in enumerate(k_mesh):
    print(f"  k[{i}] = [{k[0]:+8.4f}, {k[1]:+8.4f}, {k[2]:+8.4f}]")

K-point mesh (Cartesian, Å⁻¹):
  Shape: (8, 3)
  k[0] = [ +0.0000,  +0.0000,  +0.0000]
  k[1] = [ +0.8560,  +0.8560,  -0.8560]
  k[2] = [ +0.8560,  -0.8560,  +0.8560]
  k[3] = [ +1.7120,  +0.0000,  +0.0000]
  k[4] = [ -0.8560,  +0.8560,  +0.8560]
  k[5] = [ +0.0000,  +1.7120,  -0.0000]
  k[6] = [ -0.0000,  +0.0000,  +1.7120]
  k[7] = [ +0.8560,  +0.8560,  +0.8560]


## Lattice Vectors

The primitive lattice vectors define the unit cell geometry. The reciprocal basis (with 2π convention) determines the plane-wave grid.

In [4]:
print("Primitive lattice vectors (Å):")
for i in range(3):
    print(f"  a[{i}] = [{a_vectors[i, 0]:8.4f}, {a_vectors[i, 1]:8.4f}, {a_vectors[i, 2]:8.4f}]")

V_prim = abs(np.linalg.det(a_vectors))
print(f"\nPrimitive cell volume: {V_prim:.4f} ų")

Lx, Ly, Lz = L_size
A_super = np.vstack([Lx * a_vectors[0], Ly * a_vectors[1], Lz * a_vectors[2]])
V_super = abs(np.linalg.det(A_super))
print(f"Supercell volume ({Lx}×{Ly}×{Lz}): {V_super:.4f} ų")

Primitive lattice vectors (Å):
  a[0] = [  0.0000,   1.8350,   1.8350]
  a[1] = [  1.8350,   0.0000,   1.8350]
  a[2] = [  1.8350,   1.8350,   0.0000]

Primitive cell volume: 12.3577 ų
Supercell volume (1×1×1): 12.3577 ų


## Pair Density ρ̃

The smooth pseudo pair-density ρ̃ is the largest object. Its shape encodes the Bloch orbital structure: for each pair of k-points (k, k') and band indices (i, j), we get a 3D real-space grid.

$$\tilde{\rho}_{ki,k'j}(\mathbf{r}) = \tilde{\psi}^*_{ki}(\mathbf{r})\,\tilde{\psi}_{k'j}(\mathbf{r}) + \text{compensation charges}$$

In [5]:
print(f"Pair density shape: {rho.shape}")
print(f"  = (Nk={rho.shape[0]}, Nb={rho.shape[1]}, Nk={rho.shape[2]}, Nb={rho.shape[3]}, "
      f"Nx={rho.shape[4]}, Ny={rho.shape[5]}, Nz={rho.shape[6]})")
print(f"  dtype: {rho.dtype}")
print(f"  Memory: {rho.nbytes / 1e6:.1f} MB")
print(f"  FFT grid points: {rho.shape[4]} × {rho.shape[5]} × {rho.shape[6]} "
      f"= {np.prod(rho.shape[4:])}")

Pair density shape: (8, 5, 8, 5, 8, 8, 8)
  = (Nk=8, Nb=5, Nk=8, Nb=5, Nx=8, Ny=8, Nz=8)
  dtype: complex128
  Memory: 13.1 MB
  FFT grid points: 8 × 8 × 8 = 512


Let's look at the diagonal element ρ̃(k=0, band=0, k=0, band=0) — this is the smooth pseudo-charge density of the first Bloch orbital at the Γ point.

In [6]:
# Diagonal pair density: |ψ_{k=0,band=0}|² on the real-space grid
rho_00 = rho[0, 0, 0, 0, :, :, :]
print(f"ρ̃(k=0, i=0, k=0, j=0) shape: {rho_00.shape}")
print(f"  max |Re|: {np.max(np.abs(rho_00.real)):.6f}")
print(f"  max |Im|: {np.max(np.abs(rho_00.imag)):.6e}  (should be ~0 for diagonal)")
print(f"  Integral (sum × dV): {np.sum(rho_00.real) * V_super / np.prod(rho_00.shape):.6f}")

ρ̃(k=0, i=0, k=0, j=0) shape: (8, 8, 8)
  max |Re|: 0.230426
  max |Im|: 0.000000e+00  (should be ~0 for diagonal)
  Integral (sum × dV): 0.350793


## $C^a$ Tensors (PAW On-Site Correction)

The $C^a$ tensor for each atom $a$ encodes the on-site Coulomb correction from the PAW method. It captures the difference between all-electron and pseudo partial-wave Coulomb integrals. It has shape $(n_a, n_a, n_a, n_a)$ where $n_a$ is the number of partial-wave channels for atom $a$.

$$C^a_{i_1 i_2 i_3 i_4} = \int \left[\phi^a_{i_1}(\mathbf{r})\phi^a_{i_2}(\mathbf{r}) - \tilde{\phi}^a_{i_1}(\mathbf{r})\tilde{\phi}^a_{i_2}(\mathbf{r})\right] \frac{1}{|\mathbf{r}-\mathbf{r}'|} \left[\phi^a_{i_3}(\mathbf{r}')\phi^a_{i_4}(\mathbf{r}') - \tilde{\phi}^a_{i_3}(\mathbf{r}')\tilde{\phi}^a_{i_4}(\mathbf{r}')\right] d\mathbf{r}\,d\mathbf{r}'$$

In [7]:
print("C tensors (one per atom):")
for a, Ca in sorted(Ca_dict.items()):
    na = Ca.shape[0]
    P = na * (na + 1) // 2  # upper-triangular partial-wave pairs
    print(f"  Atom {a}: shape {Ca.shape}, na={na}, "
          f"partial-wave pairs P_a={P}, "
          f"max |C^a| = {np.max(np.abs(Ca)):.4f}")

C tensors (one per atom):
  Atom 0: shape (5, 5, 5, 5), na=5, partial-wave pairs P_a=15, max |C^a| = 0.0000


## $D^a$ Tensors (Projector Density Matrix)

The $D^a$ tensor connects Bloch orbitals to partial-wave coefficients at atom $a$. It is built from the inner products of PAW projector functions with the pseudo Bloch orbitals. It has shape $(N_k, N_b, N_k, N_b, n_a, n_a)$.

$$D^a_{ki,k'j,i_1 i_2} = \langle \tilde{p}^a_{i_1} | \tilde{\psi}_{ki} \rangle^* \langle \tilde{p}^a_{i_2} | \tilde{\psi}_{k'j} \rangle$$

where $\tilde{p}^a_i$ are the PAW projector functions.

In [8]:
print("D tensors (one per atom):")
for a, Da in sorted(Da_dict.items()):
    print(f"  Atom {a}: shape {Da.shape}, dtype={Da.dtype}")
    print(f"    max |D^a| = {np.max(np.abs(Da)):.6f}")
    # Show sparsity: what fraction of elements are near zero?
    frac_nonzero = np.count_nonzero(np.abs(Da) > 1e-10) / Da.size
    print(f"    Non-zero fraction (>1e-10): {frac_nonzero:.1%}")

D tensors (one per atom):
  Atom 0: shape (8, 5, 8, 5, 5, 5), dtype=complex128
    max |D^a| = 0.639962
    Non-zero fraction (>1e-10): 16.9%


## One-Body Matrix Elements

The one-body integrals $h_{\mathbf{k}i,\mathbf{k}'j}$ include the kinetic energy, local potential, and non-local PAW contributions. They have shape $(N_k, N_b, N_k, N_b)$.

In [9]:
h_pq = reader.h_pq
if h_pq is not None:
    unit = reader.one_body_unit or "unknown"
    print(f"One-body matrix h_pq: shape {h_pq.shape}, dtype={h_pq.dtype}")
    print(f"  Unit: {unit}")

    # Show the diagonal block h(k=0, k=0) — a Nb×Nb Hermitian matrix
    H00 = h_pq[0, :, 0, :]
    print(f"\n  h(k=0, k=0) diagonal ({unit}):")
    for i in range(H00.shape[0]):
        print(f"    band {i}: {H00[i, i].real:+.4f}")
else:
    print("One-body integrals not present in this file.")

One-body matrix h_pq: shape (8, 5, 8, 5), dtype=complex128
  Unit: Ha

  h(k=0, k=0) diagonal (Ha):
    band 0: -1.0804
    band 1: +1.9344
    band 2: +3.5872
    band 3: +3.5958
    band 4: +3.5958


## Metadata

The metadata dictionary records extraction parameters and provenance information.

In [10]:
import json
print("Metadata keys:", list(meta.keys()))
print(json.dumps(meta, indent=2, default=str))

Metadata keys: ['timestamp', 'code', 'gpaw_version', 'ase_version', 'numpy_version', 'xc', 'mode', 'ecut_eV', 'nbands_requested', 'include_compensation', 'thresholds', 'one_body', 'two_body']
{
  "timestamp": "2026-02-26T16:53:02.506848",
  "code": "GPAW+ASE",
  "gpaw_version": "25.7.0",
  "ase_version": "3.26.0",
  "numpy_version": "2.3.0",
  "xc": "unknown",
  "mode": "NoneType",
  "ecut_eV": null,
  "nbands_requested": 5,
  "include_compensation": true,
  "thresholds": {
    "rho": 0.001,
    "D": 0.01,
    "C": 0.01
  },
  "one_body": {
    "written": true,
    "unit": "Ha",
    "k_diagonal": true
  },
  "two_body": {
    "written": true,
    "unit": "Ha",
    "definition": "Eq. III.5 in position space",
    "g0": "zero",
    "k_diagonal": false,
    "split_written": false,
    "two_body_threshold": 0.0
  }
}


## Using `to_calculator_inputs()` (the Convenient Way)

In practice, you rarely need to handle the raw arrays yourself. The `to_calculator_inputs()` method packages everything into a dictionary that can be passed directly to `OneNormCalculator`.

In [11]:
reader2 = PawReader(DATA_FILE)
inputs = reader2.to_calculator_inputs()

print("Keys returned by to_calculator_inputs():")
for key, val in inputs.items():
    if isinstance(val, np.ndarray):
        print(f"  {key:12s}  ->  ndarray, shape={val.shape}, dtype={val.dtype}")
    elif isinstance(val, dict):
        # Show summary for atom dictionaries (Ca_dict, Da_dict)
        print(f"  {key:12s}  ->  dict with {len(val)} entries")
        for k2, v2 in list(val.items())[:3]:  # show first 3 atoms
            if isinstance(v2, np.ndarray):
                print(f"    [{k2}] shape={v2.shape}, dtype={v2.dtype}")
            else:
                print(f"    [{k2}] {v2}")
        if len(val) > 3:
            print(f"    ... ({len(val) - 3} more)")
    else:
        print(f"  {key:12s}  ->  {val}")

Keys returned by to_calculator_inputs():
  k_mesh        ->  ndarray, shape=(8, 3), dtype=float64
  L_size        ->  (1, 1, 1)
  a_vectors     ->  ndarray, shape=(3, 3), dtype=float64
  N_pw          ->  511
  rho           ->  ndarray, shape=(8, 5, 8, 5, 8, 8, 8), dtype=complex128
  Ca_dict       ->  dict with 1 entries
    [0] shape=(5, 5, 5, 5), dtype=float64
  Da_dict       ->  dict with 1 entries
    [0] shape=(8, 5, 8, 5, 5, 5), dtype=complex128
  h_pq          ->  ndarray, shape=(8, 5, 8, 5), dtype=complex128


## Summary

| Quantity | Shape | Physical meaning |
|----------|-------|-----------------|
| `k_mesh` | $(N_k, 3)$ | k-point coordinates in reciprocal space |
| `a_vectors` | $(3, 3)$ | Primitive lattice vectors defining the crystal |
| `rho` | $(N_k, N_b, N_k, N_b, N_x, N_y, N_z)$ | Smooth pseudo pair-density on the FFT grid |
| $C^a$ | $(n_a, n_a, n_a, n_a)$ | PAW on-site Coulomb correction (per atom) |
| $D^a$ | $(N_k, N_b, N_k, N_b, n_a, n_a)$ | Projector density matrix (per atom) |
| `h_pq` | $(N_k, N_b, N_k, N_b)$ | One-body matrix elements |

Next: **04_one_norm_calculation.ipynb** walks through the seven-step $\lambda$ computation using these quantities.

## Lazy Loading with Context Manager

For large systems where the pair density $\tilde{\rho}$ may not fit in memory, `PawReader` supports **lazy loading**. In this mode, $\tilde{\rho}$ is returned as an `h5py` dataset handle rather than a full numpy array. The `OneNormCalculator` reads one $(k, k')$ block at a time during computation.

Use the context manager (`with reader:`) to keep the HDF5 file open for the lifetime of the computation:

In [12]:
reader_lazy = PawReader(DATA_FILE)
with reader_lazy:
    inputs_lazy = reader_lazy.to_calculator_inputs(lazy=True)

    # rho is now an h5py dataset handle, not a numpy array
    rho_lazy = inputs_lazy["rho"]
    print(f"Type of rho (lazy):  {type(rho_lazy).__name__}")
    print(f"Shape:               {rho_lazy.shape}")
    print(f"Dtype:               {rho_lazy.dtype}")
    print()
    print("Other arrays are still loaded eagerly:")
    for key, val in inputs_lazy.items():
        if key == "rho":
            continue
        if isinstance(val, np.ndarray):
            print(f"  {key:12s}  ->  ndarray ({val.nbytes / 1e3:.1f} KB)")
    # The file is automatically closed when we exit the 'with' block

Type of rho (lazy):  Dataset
Shape:               (8, 5, 8, 5, 8, 8, 8)
Dtype:               complex128

Other arrays are still loaded eagerly:
  k_mesh        ->  ndarray (0.2 KB)
  a_vectors     ->  ndarray (0.1 KB)
  h_pq          ->  ndarray (25.6 KB)
